<a href="https://colab.research.google.com/github/ryanatberkeley/aeesp-grand-challenges-nlp-pipeline/blob/main/bertopic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install necessary libraries
try:
    from bertopic import BERTopic
except ImportError:
    !pip install bertopic
    from bertopic import BERTopic

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip install sentence_transformers
    from sentence_transformers import SentenceTransformer

import pandas as pd
import glob
from google.colab import files # Used for downloading the final CSV to your computer

# load and merge data
# need to upload csv files by yourself
# (if using Google Drive, change this to: glob.glob('/content/drive/MyDrive/YOUR_FOLDER/works_df_*.csv'))
file_paths = glob.glob('works_df_*.csv')

if not file_paths:
    print("⚠️ ERROR: No CSV files found! Please upload them to Colab or check your folder path.")
else:
    dfs = [pd.read_csv(file) for file in file_paths]
    df = pd.concat(dfs, ignore_index=True)

    print(f"Total papers initially loaded: {len(df)}")

    # clean Data (filter by year and remove missing abstracts)
    df = df[df['publication_year'] >= 2000]
    pre_clean_count = len(df)
    df = df.dropna(subset=['abstract'])
    df = df[df['abstract'] != 'No abstract available']

    print(f"Removed {pre_clean_count - len(df)} papers due to missing or incoherent abstracts.")
    print(f"Final usable papers for modeling: {len(df)}")

    # prep text for BERTopic
    df['title'] = df['title'].fillna('')
    df['keywords'] = df['keywords'].fillna('')
    df['combined_text'] = df['title'] + " " + df['keywords'] + " " + df['abstract']

    docs = df['combined_text'].tolist()

    # initialize the SPECTER 2 model for scientific embeddings
    print("Downloading and loading SPECTER2. This might take a minute...")
    specter_model = SentenceTransformer("allenai/specter2_base")

    # define the grand challenge topics (Zero-Shot Topic Modeling)
    print("Initializing Guided BERTopic model with SPECTER2 for Grand Challenge 1...")

    grand_challenges = [
        "Energy efficient water conservation strategies and technologies: energy-efficient water systems, water conservation, rainwater harvesting, smart water systems, sustainable water management, water reuse, water recovery, stormwater reuse, graywater treatment, stakeholder input, behavioral science, social science, water demand management, leak detection, water loss control, smart metering, water pricing, drip irrigation, precision irrigation, urban water management, integrated urban water management, nonpotable reuse",

        "Low-cost desalination and water reuse technologies: water reuse, water reclamation, water recovery, wastewater recovery, reclaimed water use, desalination, brackish water treatment, membrane treatment, membrane separation, high-pressure membrane filtration, advanced membranes, polymers, reverse osmosis, nanofiltration, electrodialysis, brine management, concentrate minimization, zero liquid discharge, ZLD, solar desalination, solar distillation, forward osmosis, membrane distillation, capacitive deionization, ion exchange, advanced oxidation processes, UV advanced oxidation, hydroxyl radicals, AOPs, brine valorization, circular water systems, energy-water nexus",

        "Water supply and water quality forecasting tools: sensors, sensor networks, water reliability, water supply, water infrastructure, water security, smart phone, wireless, pathogens, water quality monitoring, biosensors, remote sensing, Internet of Things, smart water networks, early warning systems, contamination event detection, predictive modeling, machine learning, artificial intelligence, digital twin, anomaly detection, decision support systems, spatiotemporal analysis, hydrologic modeling, hydraulic modeling, drought forecasting",

        "Energy-neutral or energy-positive cost-effective wastewater treatment technologies: water recovery, water reuse, pathogens, low cost, microbial fuel cells, anaerobic digestion, biogas production, primary wastewater treatment, secondary wastewater treatment, nutrient recovery, phosphorus recovery, nitrogen recovery, biosolids, sludge treatment, anaerobic membrane bioreactor, circular sanitation, decentralized sanitation, onsite sanitation, nature-based treatment, constructed wetlands, lagoon systems, micropollutant removal, pharmaceutical removal, trace contaminant removal",

        "Water, sanitation, and hygiene challenges in low-income countries: user acceptance, stakeholders, no water toilets, onsite water treatment, decentralized water treatment, point of use water treatment, behavioral science, social science, policy, WASH, basic sanitation, sanitation access, hygiene access, fecal sludge management, septic systems, household water treatment, chlorination, solar disinfection, hand hygiene, hygiene behavior",

        "Aging water infrastructure: water infrastructure, water distribution systems, lead pipes, adaptive systems, aging infrastructure, asset management, pipe failure, water main breaks, corrosion control, copper corrosion, biofilm, nitrification, hydraulic transients, water age, leakage, service reliability, risk-based prioritization, infrastructure resilience, pipe replacement, smart infrastructure"
    ]

    # run BERTopic with the new embedding model
    topic_model = BERTopic(
        embedding_model=specter_model,
        zeroshot_topic_list=grand_challenges,
        zeroshot_min_similarity=0.5,
        language="english",
        calculate_probabilities=True,
        verbose=True
    )

    topics, probabilities = topic_model.fit_transform(docs)

    # display results in notebook
    topic_info = topic_model.get_topic_info()
    display(topic_info.head(15))

    # save and download the Data
    # appended the assigned AI topics directly to our master dataframe
    df['assigned_topic_id'] = topics

    # save the dataframe to a CSV file
    csv_filename = 'grand_challenges_classified_papers.csv'
    df.to_csv(csv_filename, index=False)
    print(f"✅ Success! Saving results to '{csv_filename}' and triggering download...")

    # prompt the browser to download the file
    files.download(csv_filename)

In [ ]:
from google.colab import files

# 1. Save the topic table to a CSV file in the Colab environment
topic_info.to_csv('bertopic_summary.csv', index=False)

# 2. Trigger the download to your local computer
files.download('bertopic_summary.csv')